In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [17]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# Milestone 1

In [18]:
print(train.columns)

Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object')


In [19]:
# Answer 1
freq = train['answer'].value_counts()
print(freq)
print(f'Most Frequent: {freq.max()}')
print(f'least Frequent: {freq.min()}')
print("Sum:", freq.min() + freq.max())

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Most Frequent: 490
least Frequent: 324
Sum: 814


In [20]:
# Answer 2
import string
prompt_lowercase = train["prompt"].str.lower()

prompts = prompt_lowercase.str.translate(str.maketrans('', '', string.punctuation))

vocab = set()

for text in prompts:
    vocab.update(text.split())

print(len(vocab))

859


In [21]:
# Answer 3
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string

prompt = train.loc[train["id"] == 1, "prompt"].iloc[0]

prompt = prompt.lower()

prompt = prompt.translate(
    str.maketrans('', '', string.punctuation)
)

words = prompt.split()

filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]

print(len(filtered_words))

13


In [22]:
# Answer 4
from sklearn.feature_extraction.text import TfidfVectorizer

combined_text = (
    
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(combined_text)

print(X.shape[1])

2762


In [23]:
# Answer 5
from sklearn.metrics.pairwise import cosine_similarity

row = train[train["id"] == 1].iloc[0]

prompt_vec = vectorizer.transform([row["prompt"]])
option_a_vec = vectorizer.transform([row["A"]])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print(round(similarity, 4))

0.272


In [24]:
# Answer 6
from sklearn.metrics.pairwise import cosine_similarity

correct = 0
total = len(train)

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    sims = {}

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sims[option] = cosine_similarity(prompt_vec, option_vec)[0][0]

    predicted = max(sims, key=sims.get)

    if predicted == row["answer"]:
        correct += 1

accuracy = (correct / total) * 100

print(round(accuracy, 2))

13.55


In [25]:
# ans 7. 1.0
# ans 8 0.5
# ans 9 0.4213



In [26]:
# ansswer 10
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

def map3_score(actual, preds):
    if actual in preds[:3]:
        return 1 / (preds.index(actual) + 1)
    return 0

scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    sims = {}

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sims[option] = cosine_similarity(prompt_vec, option_vec)[0][0]

    preds = sorted(sims, key=sims.get, reverse=True)

    scores.append(map3_score(row["answer"], preds))

print(round(sum(scores) / len(scores), 5))

0.29617
